# Lab: Orpheus ← Stage 1–2 synthesis manifest

**Goal:** Thử **Orpheus finetune-prod** (open-weight, fixed voice names) với **cùng contract** Stage 1–2 (`spoken_text`, `voice_profile_id`).

**Không phải:** API trả phí (`pip install orpheus-tts` client) · chọn TTS cuối · sửa script gốc · production adapter

**Runtime:** Colab GPU — **A100** khuyến nghị (model ~3B + vLLM). T4 có thể chật/OOM.

**Upstream:** [canopyai/Orpheus-TTS](https://github.com/canopyai/Orpheus-TTS) · model `canopylabs/orpheus-tts-0.1-finetune-prod` · voices: `tara, leah, jess, leo, dan, mia, zac, zoe`


## 0. Config repo


In [ ]:
REPO_URL = "https://github.com/phamtu2x5/ielts-script2audio.git"
BRANCH = "main"
WORKDIR = "/content/ielts-script2audio"


## 1. Clone / pull repo


In [ ]:
import os
from pathlib import Path

if not Path(WORKDIR).exists():
    !git clone --branch {BRANCH} {REPO_URL} {WORKDIR}
else:
    %cd {WORKDIR}
    !git pull
%cd {WORKDIR}
print("CWD:", os.getcwd())


## 2. Install control-plane + Orpheus (local open-weight)

**Không** cài `orpheus-tts` (API client có key).

Orpheus local dùng **vLLM**. Lỗi hay gặp trên Colab:

`libcudart.so.13: cannot open shared object file`

→ Wheel vLLM mới build cho **CUDA 13**, trong khi Colab thường chỉ có runtime **CUDA 12.x**.

**Cách xử lý (upstream Orpheus):** pin `vllm==0.7.3` sau khi cài `orpheus-speech`.

Runtime: **A100** tốt hơn; T4 có thể OOM.


In [ ]:
# --- light project package ---
!pip -q install -e ".[dev]" soundfile

# --- Orpheus local stack (NOT the paid API client "orpheus-tts") ---
# 1) install orpheus-speech (pulls a vLLM)
# 2) force older vLLM known to work with Orpheus + Colab CUDA 12
!pip -q uninstall -y orpheus-tts 2>/dev/null || true
!pip -q install "orpheus-speech"
!pip -q install "vllm==0.7.3"

import os
import torch

print("torch:", torch.__version__)
print("torch.cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("vram_gb:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

# Show which libcudart is visible (debug)
!ldconfig -p 2>/dev/null | grep -i cudart | head -n 20 || true
!ls -la /usr/local/cuda/lib64/libcudart.so* 2>/dev/null || true
!ls -la /usr/local/cuda-*/lib64/libcudart.so* 2>/dev/null | head -n 20 || true

# Import smoke — this is where libcudart.so.13 often blows up
try:
    import vllm
    print("vllm:", getattr(vllm, "__version__", "unknown"))
    from orpheus_tts import OrpheusModel
    print("OrpheusModel import: OK")
except Exception as e:
    print("IMPORT FAILED:", type(e).__name__, e)
    print(
        """
Fix hints (Colab):
1) Runtime → Change runtime type → GPU (A100 if possible), then Runtime → Restart session
2) Re-run THIS install cell after restart (clean kernel)
3) If still libcudart.so.13:
   !pip -q uninstall -y vllm
   !pip -q install "vllm==0.7.3"
4) Official Orpheus Colab (reference install):
   https://colab.research.google.com/drive/1KhXT56UePPUHhqitJNUxq63k-pQomz3N
5) Do not continue to §4 until import succeeds
"""
    )


## 3. Stage 1–2: prepare manifest (cùng pipeline Kokoro lab)

Dùng **cùng** `examples/part1_valid.json` → cùng `spoken_text` để so với Kokoro công bằng.


In [ ]:
import json
from pathlib import Path

!mkdir -p outputs lab_audio/orpheus_part1
!ielts-s2a prepare examples/part1_valid.json --pretty -o outputs/part1_manifest.json

m = json.loads(Path("outputs/part1_manifest.json").read_text())
print("transcript_id:", m["transcript_id"])
print("valid:", m["validation"]["valid"])
print("n_requests:", len(m["requests"]))
for u in m["prepared_utterances"]:
    if u["event_id"] in {"e004", "e006", "e008", "e011"}:
        print(u["event_id"], "DISPLAY:", u["display_text"])
        print("      SPOKEN :", u["spoken_text"])


## 4. Render full Part 1 dialogue (2 voices)

Map: `configs/voice_maps/orpheus_en_part1.json`  
- `vp_spk_a` → **tara**  
- `vp_spk_b` → **leo**  

Dùng `spoken_text` Stage 2. Có thể lâu (load model + 15 segments).

Smoke nhanh: thêm `--limit 4` vào lệnh python bên dưới.


In [ ]:
import json
import os
from pathlib import Path

OUT_DIR = Path("lab_audio/orpheus_part1")
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT = OUT_DIR / "lab_render_report.json"

# Smoke first (4 segments). Remove --limit 4 for full dialogue.
cmd = (
    "python scripts/lab_render_orpheus_from_manifest.py "
    "--manifest outputs/part1_manifest.json "
    "--voice-map configs/voice_maps/orpheus_en_part1.json "
    "--out-dir lab_audio/orpheus_part1 "
    "--temperature 0.6 "
    "--repetition-penalty 1.3 "
    "--limit 4"
)
print("Running:\n", cmd)
ret = os.system(cmd)
print("render exit code:", ret)

if not REPORT.is_file():
    raise FileNotFoundError(
        f"{REPORT} was not created.\n"
        "Render failed earlier (often Orpheus/vLLM/CUDA import).\n"
        "Scroll up for 'libcudart.so.13' or OrpheusModel errors.\n"
        "Fix: §2 install with vllm==0.7.3 → Runtime Restart → re-run §2, §3, §4.\n"
        "Official reference Colab: https://colab.research.google.com/drive/1KhXT56UePPUHhqitJNUxq63k-pQomz3N"
    )

report = json.loads(REPORT.read_text())
print("ok:", report["ok_count"], "fail:", report["fail_count"])
print("model:", report.get("model_name"))
for s in report["segments"]:
    print(s["segment_id"], s["status"], s.get("backend_voice_id"), s.get("error") or "")


## 5. Tracking (một cách duy nhất)

Với **từng** segment:

1. Đọc **DISPLAY** (script) và **SPOKEN** (text đưa TTS).
2. Play audio.
3. Ghi `yes` / `partial` / `no` ở cell §5b.

So với Kokoro: cùng nội dung Stage-2, khác engine + voice names.


In [ ]:
from pathlib import Path
from IPython.display import Audio, display
import json

REPORT_PATH = Path("lab_audio/orpheus_part1/lab_render_report.json")
report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
segments = report.get("segments") or []
audio_dir = REPORT_PATH.parent

print(f"Transcript: {report.get('transcript_id')} | segments: {len(segments)}")
print(f"Backend: {report.get('backend')} | model: {report.get('model_name')}")
print()

for i, seg in enumerate(segments, start=1):
    fname = seg.get("output_filename") or (Path(seg["output"]).name if seg.get("output") else None)
    wav = audio_dir / fname if fname else None
    print("=" * 72)
    print(
        f"[{i}/{len(segments)}] {seg.get('segment_id')} | speaker={seg.get('speaker_id')} | "
        f"voice={seg.get('backend_voice_id')} | status={seg.get('status')}"
    )
    print(f"DISPLAY : {seg.get('display_text', '')}")
    print(f"SPOKEN  : {seg.get('spoken_text', '')}")
    if seg.get("protected_region_ids"):
        print(f"protected: {seg.get('protected_region_ids')}")
    if seg.get("error"):
        print(f"ERROR   : {seg.get('error')}")
    if wav is not None and wav.is_file():
        display(Audio(filename=str(wav)))
    else:
        print(f"(missing wav: {fname})")
    print()


## 5b. Ghi tracking

Điền `content_match` rồi chạy cell để lưu `segment_review_filled.json`.


In [ ]:
from pathlib import Path
import json

report_path = Path("lab_audio/orpheus_part1/lab_render_report.json")
report = json.loads(report_path.read_text(encoding="utf-8"))

reviews = {
    seg["segment_id"]: {"content_match": "", "notes": ""}
    for seg in (report.get("segments") or [])
    if seg.get("segment_id")
}

# Example:
# reviews["seg_0004"]["content_match"] = "partial"
# reviews["seg_0004"]["notes"] = "spelling still rushed"

filled = []
missing = []
for seg in report.get("segments") or []:
    sid = seg.get("segment_id")
    human = reviews.get(sid) or {}
    match = (human.get("content_match") or "").strip().lower()
    if match not in {"yes", "partial", "no"}:
        missing.append(sid)
    filled.append(
        {
            "segment_id": sid,
            "event_id": seg.get("event_id"),
            "speaker_id": seg.get("speaker_id"),
            "backend_voice_id": seg.get("backend_voice_id"),
            "output_filename": seg.get("output_filename"),
            "display_text": seg.get("display_text"),
            "spoken_text": seg.get("spoken_text"),
            "status": seg.get("status"),
            "content_match": match,
            "notes": human.get("notes", ""),
        }
    )

out = Path("lab_audio/orpheus_part1/segment_review_filled.json")
out.write_text(json.dumps(filled, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Wrote", out)
if missing:
    print("Chưa chấm:", ", ".join(missing))
for row in filled:
    print(row["segment_id"], row["content_match"] or "(empty)", "|", (row["display_text"] or "")[:50])


## 6. Ghép file liền (tuỳ chọn)

Dùng chung script adaptive gap như lab Kokoro.


In [ ]:
import json
from pathlib import Path
from IPython.display import Audio, display

!python scripts/lab_concat_segment_wavs.py \
  --report lab_audio/orpheus_part1/lab_render_report.json \
  --out lab_audio/orpheus_part1/part1_full.wav \
  --gap-mode adaptive \
  --same-speaker-gap-ms 220 \
  --turn-gap-ms 520

full = Path("lab_audio/orpheus_part1/part1_full.wav")
meta = json.loads(Path("lab_audio/orpheus_part1/part1_full.concat.json").read_text())
print(meta.get("duration_sec"), "sec,", meta.get("num_segments_used"), "segments")
if full.is_file():
    display(Audio(filename=str(full)))


## 7. Survey nhiều giọng Orpheus (cùng dòng Stage-2)

Mặc định preset **`en_shortlist`**: `tara, leah, leo, dan` (nhẹ hơn full 8).  
Full 8 voices: đổi `--preset en_all` (lâu hơn).

Cùng event: e004 spelling · e006 postcode · e008 phone · e011 fee.


In [ ]:
from pathlib import Path
from IPython.display import Audio, display
import json
from collections import defaultdict

!python scripts/lab_survey_orpheus_voices.py \
  --manifest outputs/part1_manifest.json \
  --inventory configs/voice_maps/orpheus_voice_inventory.json \
  --preset en_shortlist \
  --out-dir lab_audio/orpheus_voice_survey \
  --event-ids e004,e006,e008,e011 \
  --end-pad-ms 450

report = json.loads(Path("lab_audio/orpheus_voice_survey/voice_survey_report.json").read_text())
print("voices:", report.get("voices"))
print("ok/fail:", report.get("ok_count"), report.get("fail_count"))

by_event = defaultdict(list)
for r in report.get("renders") or []:
    by_event[r.get("event_id")].append(r)

for event_id, items in by_event.items():
    items = sorted(items, key=lambda x: x.get("voice_id") or "")
    sample = items[0] if items else {}
    print("=" * 72)
    print(f"LINE {event_id}")
    print(f"DISPLAY : {sample.get('display_text')}")
    print(f"SPOKEN  : {sample.get('spoken_text')}")
    for it in items:
        print(f"--- voice={it.get('voice_id')} status={it.get('status')}")
        if it.get("error"):
            print("ERROR", it["error"])
            continue
        wav = Path("lab_audio/orpheus_voice_survey") / it["output_filename"]
        if wav.is_file():
            display(Audio(filename=str(wav)))
        else:
            print("missing", wav)


## 8. Checklist so với Kokoro

1. Install/load Orpheus trên Colab có ổn không (OOM? vLLM?)?
2. 2 speaker (tara/leo) có tách giọng rõ hơn Kokoro không?
3. Spelling / postcode / phone: rõ hơn, kém hơn, hay vẫn nuốt cuối?
4. Giọng có “sống” hơn (ít flat) không?
5. Có phải sửa script không? (**Không** — chỉ báo cáo.)

**Verdict lab Orpheus:** Pass / Borderline / Fail — vẫn **`not_selected`** cho production.

Ghi note ngắn (optional): `docs/research/lab/notes-orpheus-part1.md` sau khi pull về máy.
